# MF-IA baseline (input-augmented multi-fidelity) on the four benchmarks

MF-IA: the low-fidelity response is an **input** and the network predicts the **high-fidelity response directly** (Guo et al., 2022; Conti et al., 2023). It is the Arm 3 script with the residual target switched off (`MFIA=1`): same channels, backbone, loss, epochs, seeds and splits. 4 systems x 5 seeds = 20 MF-IA runs (about 1.5-2 h on a T4), then the same 20 for Arm 3 so the last cell can pair them by seed (another 1.5-2 h). Results go to `colab_benchmark/results/`.

**Needs in `MyDrive/colab_benchmark/`:** `data/` (4 `.npz` databases) and `scripts/` (from `FINAL_SUBMISSION/5_colab_benchmark/`). Re-running skips finished runs.

Run all. If the session dies, reconnect and Run all again.

In [ ]:
import torch, platform
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all.")
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| python", platform.python_version())

In [ ]:
import os, shutil

from google.colab import drive
drive.mount("/content/drive")

# ── edit this one line if you put the folder somewhere else ──
BASE = "/content/drive/MyDrive/colab_benchmark"

OUTD = os.path.join(BASE, "results")
os.makedirs(OUTD, exist_ok=True)

# The release scripts read their databases from the working directory by name, so stage
# everything into one local folder and run there.
WORK = "/content/bench"
os.makedirs(WORK, exist_ok=True)
for sub in ("data", "scripts"):
    src = os.path.join(BASE, sub)
    assert os.path.isdir(src), f"missing {src}"
    for f in os.listdir(src):
        d = os.path.join(WORK, f)
        if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(os.path.join(src, f)):
            shutil.copy2(os.path.join(src, f), d)
os.chdir(WORK)
print("staged:", sorted(os.listdir(WORK)))
print("results ->", OUTD)

## Patch the staged Arm 3 scripts with the MF-IA switch

In [ ]:
# MF-IA = the Arm 3 script with MFIA=1: same inputs (LF response included), same network, loss,
# seeds and splits, but the target is the HF response itself and the LF solution is NOT added back.
# The patch is a no-op for Arm 3 (MFIA defaults to 0). Applied to the staged copies in WORK only.
FLAG = 'MFIA = __import__("os").environ.get("MFIA", "0") == "1"   # MF-IA: LF is an input only; target = u_HF\n'
SEED_LINE = 'SEED = int(__import__("os").environ.get("SEED", 42))\n'
PATCHES = {
 "sdof_arm3.py": [
  ("delta_u     = u_nl_raw - u_lin_raw ", "delta_u     = u_nl_raw - (0.0 if MFIA else u_lin_raw)"),
  ("    pred_u = u_lin_raw[sample_indices] + pred_delta_u", "    pred_u = (0.0 if MFIA else u_lin_raw[sample_indices]) + pred_delta_u")],
 "tunnel_arm3.py": [
  ("residual_raw = (u_nl_raw - u_red_raw) ", "residual_raw = (u_nl_raw - (0.0 if MFIA else u_red_raw))"),
  ("    pred_u     = u_red_raw[sample_indices] + pred_delta ", "    pred_u     = (0.0 if MFIA else u_red_raw[sample_indices]) + pred_delta ")],
}
PATCHES["turbine_arm3.py"] = PATCHES["tunnel_arm3.py"]
for f, reps in PATCHES.items():
    s = open(f).read()
    if "MFIA = " in s: print("already patched", f); continue
    for a, b in reps:
        assert s.count(a) == 1, (f, a); s = s.replace(a, b)
    assert s.count(SEED_LINE) == 1, f; s = s.replace(SEED_LINE, SEED_LINE + FLAG)
    open(f, "w").write(s); print("patched", f)

# frame: ARM 4 = Arm 3 input channels (u_red, dlin) + Arm 2 target u_nl and reconstruction
s = open("frame_run.py").read()
A = "exec(code[6], G)          # channels\nexec(code[7], G)          # train + rollout\n"
B = ("for _k in (6, 7):\n"
     "    assert code[_k].count(\"if arm == 3 else []\") == 1, _k\n"
     "    code[_k] = code[_k].replace(\"if arm == 3 else []\", \"if arm in (3, 4) else []\")\n")
if B not in s:
    assert s.count(A) == 1; s = s.replace(A, B + A); open("frame_run.py", "w").write(s); print("patched frame_run.py")
import py_compile
for f in ("sdof_arm3.py", "tunnel_arm3.py", "turbine_arm3.py", "frame_run.py"): py_compile.compile(f, doraise=True)
print("all compile")


## Configuration

In [ ]:
SEEDS = [42, 0, 1, 2, 3]
ARMS  = (4, 3)        # 4 = MF-IA first, then Arm 3 re-run on the same seeds for the paired table
SPEC = {
    ("sdof", 3):    dict(cmd=["python3", "sdof_arm3.py"],
                         env={"DB": "sdof_eta60_n1500.npz"}, read="sdof3"),
    ("frame3", 3):  dict(cmd=["python3", "frame_run.py", "{seed}", "3", "10"],
                         env={}, read="frame"),
    ("tunnel", 3):  dict(cmd=["python3", "tunnel_arm3.py"],
                         env={"N_MAX": "500", "TAG": "bench_tunnel_arm3_s{seed}"},
                         read="csv", csv="bench_tunnel_arm3_s{seed}_results.csv",
                         metric="unweighted"),
    ("turbine", 3): dict(cmd=["python3", "turbine_arm3.py"],
                         env={"DB_PATH": "wt_v20_full.npz", "N_MAX": "0", "FLIP": "1",
                              "STEP": "50", "SAMPLE_W": "0",
                              "TAG": "bench_wt_arm3_s{seed}"},
                         read="csv", csv="bench_wt_arm3_s{seed}_results.csv",
                         metric="unweighted"),
    ("sdof", 4):    dict(cmd=["python3", "sdof_arm3.py"],
                         env={"DB": "sdof_eta60_n1500.npz", "MFIA": "1"}, read="sdof3"),
    ("frame3", 4):  dict(cmd=["python3", "frame_run.py", "{seed}", "4", "10"],
                         env={}, read="frame"),
    ("tunnel", 4):  dict(cmd=["python3", "tunnel_arm3.py"],
                         env={"N_MAX": "500", "MFIA": "1", "TAG": "bench_tunnel_mfia_s{seed}"},
                         read="csv", csv="bench_tunnel_mfia_s{seed}_results.csv",
                         metric="unweighted"),
    ("turbine", 4): dict(cmd=["python3", "turbine_arm3.py"],
                         env={"DB_PATH": "wt_v20_full.npz", "N_MAX": "0", "FLIP": "1",
                              "STEP": "50", "SAMPLE_W": "0", "MFIA": "1",
                              "TAG": "bench_wt_mfia_s{seed}"},
                         read="csv", csv="bench_wt_mfia_s{seed}_results.csv",
                         metric="unweighted"),
}
SYSTEMS = ["sdof", "frame3", "turbine", "tunnel"]
print(f"{len(SYSTEMS)} systems x {len(ARMS)} arms x {len(SEEDS)} seeds = {len(SYSTEMS)*len(ARMS)*len(SEEDS)} runs")


## Run (resumable)

In [ ]:
import csv, json, os, re, subprocess, time
import statistics as st

def key(ex, arm, seed):
    return f"{ex}_arm{arm}_s{seed}"

def finished(ex, arm, seed):
    jf = os.path.join(OUTD, key(ex, arm, seed) + ".json")
    if not os.path.exists(jf):
        return False
    try:
        r = json.load(open(jf))
        return "arm" in r and "arm1" in r
    except Exception:
        print(f"   corrupt {os.path.basename(jf)} -> will re-run"); return False

def save_atomic(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=1); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)

def read_result(spec, seed, log):
    """-> (arm1, armN). Four scripts, four reporting conventions."""
    how = spec["read"]
    if how == "sdof3":
        # "  Global R2                      0.7335          0.9250"
        m = re.search(r"Global R.{0,3}\s+([+-]?\d+\.\d+)\s+([+-]?\d+\.\d+)", log)
        assert m, "SDOF arm3: could not find the Global R2 line"
        return float(m.group(1)), float(m.group(2))
    if how == "sdof2":
        # "  SDOF u: R2=0.3736 peak%=... " ; Arm 1 comes from the arm3 run of the same seed
        m = re.search(r"SDOF\s+u:\s*R2=([+-]?\d+\.\d+)", log)
        assert m, "SDOF arm2: could not find the R2 line"
        return None, float(m.group(1))
    if how == "frame":
        m = re.search(r"SUMMARY seed=\S+ arm1=([+-]\d+\.\d+) arm\d=([+-]\d+\.\d+)", log)
        assert m, "frame: could not find the SUMMARY line"
        return float(m.group(1)), float(m.group(2))
    if how == "csv":
        path = spec["csv"].format(seed=seed)
        rows = {r["label"]: r for r in csv.DictReader(open(path))}
        def val(r):
            if spec.get("metric") == "unweighted":
                return st.mean([float(r[f"r2_s{i}"]) for i in range(1, 12)
                                if f"r2_s{i}" in r])
            return float(r["r2_overall"])
        return val(rows["u_red_ref"]), val(rows["trained"])
    raise ValueError(how)

todo = [(e, a, s) for a in ARMS for e in SYSTEMS for s in SEEDS]   # all MF-IA first
pre = [t for t in todo if finished(*t)]
print(f"{len(pre)}/{len(todo)} already done, {len(todo)-len(pre)} to go\n")
t_start = time.perf_counter(); n_new = 0

for n, (ex, arm, seed) in enumerate(todo, 1):
    tag = f"{ex}/arm{arm}/s{seed}"
    if finished(ex, arm, seed):
        print(f"[{n:2d}/{len(todo)}] {tag:22s} SKIP (done)"); continue
    spec = SPEC[(ex, arm)]
    cmd = [c.format(seed=seed) for c in spec["cmd"]]
    env = {**os.environ, "SEED": str(seed), "MPLBACKEND": "Agg",
           **{k: v.format(seed=seed) for k, v in spec["env"].items()}}
    print(f"[{n:2d}/{len(todo)}] {tag:22s} {' '.join(cmd)} ...", flush=True)
    t0 = time.perf_counter()
    p = subprocess.run(cmd, env=env, capture_output=True, text=True)
    log = p.stdout + "\n" + p.stderr
    open(os.path.join(OUTD, key(ex, arm, seed) + ".log"), "w").write(log)
    if p.returncode != 0:
        print(f"   !! exit {p.returncode}; see {key(ex,arm,seed)}.log\n"); continue
    try:
        a1, aN = read_result(spec, seed, log)
    except Exception as exc:
        print(f"   !! could not parse: {exc}\n"); continue
    save_atomic(dict(example=ex, arm=arm, seed=seed, arm1=a1, armN=aN,
                     sec=round(time.perf_counter() - t0, 1)),
                os.path.join(OUTD, key(ex, arm, seed) + ".json"))
    n_new += 1
    el = time.perf_counter() - t_start
    print(f"   arm1={a1 if a1 is None else round(a1,4)}  arm{arm}={aN:+.4f}  "
          f"{time.perf_counter()-t0:.0f}s | ETA "
          f"{(len(todo)-len(pre)-n_new)*el/n_new/60:.0f}m\n", flush=True)

nd = sum(finished(*t) for t in todo)
print(f"{nd}/{len(todo)} complete. {(time.perf_counter()-t_start)/60:.0f} min this session.")
if nd < len(todo):
    print("Not finished -- re-run cells 1-3 then this cell to continue.")

## Paired table

In [ ]:
import csv, glob, json, os
import statistics as st

R = {}
for f in glob.glob(os.path.join(OUTD, "*_arm*_s*.json")):
    r = json.load(open(f)); R[(r["example"], r["arm"], r["seed"])] = r
NICE = {"sdof": "SDOF oscillator", "frame3": "3-storey steel frame",
        "turbine": "Wind turbine on a monopile", "tunnel": "Buried tunnel"}
rows = [["system", "seed", "arm1", "arm2", "arm3", "mfia", "arm1_from_mfia_run"]]
summ = [["system", "n", "arm1", "arm2", "arm3", "mfia", "mfia_minus_arm3", "mfia_minus_arm2"]]
fmt = lambda v: f"{st.mean(v):+.4f} +/- {st.stdev(v):.4f}"
for ex in SYSTEMS:
    per = []
    for s in SEEDS:
        g = lambda a, k="armN": R[(ex, a, s)][k] if (ex, a, s) in R else None
        row = [ex, s, g(3, "arm1"), g(2), g(3), g(4), g(4, "arm1")]
        rows.append(row)
        if None not in (row[2], row[4], row[5]): per.append(row)   # Arm 2 optional
    if len(per) < 2:
        print(f"{NICE[ex]:28} incomplete ({len(per)}/{len(SEEDS)})"); continue
    col = lambda k: [r[k] for r in per]
    d3 = [r[5] - r[4] for r in per]; d2 = [r[5] - r[3] for r in per if r[3] is not None]
    summ.append([ex, len(per)] + [fmt([r[k] for r in per if r[k] is not None]) if sum(r[k] is not None for r in per) > 1 else '-' for k in (2, 3, 4, 5)] + [fmt(d3), fmt(d2) if len(d2) > 1 else '-'])
    same_split = all(r[6] is None or abs(r[6] - r[2]) < 1e-4 for r in per)
    print(f"{NICE[ex]:28} n={len(per)}  A1 {fmt(col(2))}  A3 {fmt(col(4))}  "
          f"MF-IA {fmt(col(5))}  MF-IA-A3 {fmt(d3)}  same split as Arm 3: {same_split}")
for name, tab in (("mfia_per_seed.csv", rows), ("table2_mfia.csv", summ)):
    with open(os.path.join(OUTD, name), "w", newline="") as fh: csv.writer(fh).writerows(tab)
    print("wrote", os.path.join(OUTD, name))
